In [1]:
import pandas as pd
import numpy as np

In [2]:
matches = pd.read_csv("../data/processed/matches_cleaned.csv")

matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)

print("Shape:", matches.shape)
matches.head()

Shape: (891, 20)


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335987,2007/08,Jaipur,2008-04-21,League,SR Watson,Sawai Mansingh Stadium,Rajasthan Royals,Punjab Kings,Punjab Kings,bat,Rajasthan Royals,wickets,6.0,167.0,20.0,N,NaN,Aleem Dar,RB Tiffin


In [3]:
matches['team1'] = matches['team1'].str.strip().str.lower()
matches['team2'] = matches['team2'].str.strip().str.lower()
matches['winner'] = matches['winner'].str.strip().str.lower()

team_map = {
    "delhi daredevils": "delhi capitals",
    "kings xi punjab": "punjab kings",
    "rising pune supergiant": "rising pune supergiants"
}

matches['team1'] = matches['team1'].replace(team_map)
matches['team2'] = matches['team2'].replace(team_map)
matches['winner'] = matches['winner'].replace(team_map)

print("Team names cleaned")

Team names cleaned


In [4]:
matches['team1_win'] = (matches['winner'] == matches['team1']).astype(int)

matches[['team1','team2','winner','team1_win']].head()

,team1,team2,winner,team1_win
0,royal challengers bangalore,kolkata knight riders,kolkata knight riders,0
1,punjab kings,chennai super kings,chennai super kings,0
2,delhi capitals,rajasthan royals,delhi capitals,1
3,mumbai indians,royal challengers bangalore,royal challengers bangalore,0
4,rajasthan royals,punjab kings,rajasthan royals,1


In [5]:
team_stats = {}
recent_matches = {}
head2head = {}

matches['team1_win_rate'] = 0.5
matches['team2_win_rate'] = 0.5
matches['team1_recent_form'] = 0.5
matches['team2_recent_form'] = 0.5
matches['head_to_head'] = 0.0

for i in range(len(matches)):
    
    team1 = matches.loc[i, 'team1']
    team2 = matches.loc[i, 'team2']
    winner = matches.loc[i, 'winner']
    
    for team in [team1, team2]:
        if team not in team_stats:
            team_stats[team] = [0, 0]
        if team not in recent_matches:
            recent_matches[team] = []
    
    t1_wins, t1_matches = team_stats[team1]
    t2_wins, t2_matches = team_stats[team2]
    
    matches.loc[i, 'team1_win_rate'] = t1_wins / t1_matches if t1_matches > 0 else 0.5
    matches.loc[i, 'team2_win_rate'] = t2_wins / t2_matches if t2_matches > 0 else 0.5
    
    def get_form(results):
        if len(results) == 0:
            return 0.5
        return sum(results[-5:]) / len(results[-5:])
    
    matches.loc[i, 'team1_recent_form'] = get_form(recent_matches[team1])
    matches.loc[i, 'team2_recent_form'] = get_form(recent_matches[team2])
    
    pair = tuple(sorted([team1, team2]))
    
    if pair not in head2head:
        head2head[pair] = {}
    
    t1_h2h = head2head[pair].get(team1, 0)
    t2_h2h = head2head[pair].get(team2, 0)
    
    matches.loc[i, 'head_to_head'] = t1_h2h - t2_h2h
    
    # UPDATE AFTER FEATURE CALCULATION
    
    team_stats[team1][1] += 1
    team_stats[team2][1] += 1
    
    if winner == team1:
        team_stats[team1][0] += 1
        recent_matches[team1].append(1)
        recent_matches[team2].append(0)
    elif winner == team2:
        team_stats[team2][0] += 1
        recent_matches[team2].append(1)
        recent_matches[team1].append(0)
    
    if winner not in head2head[pair]:
        head2head[pair][winner] = 0
    
    head2head[pair][winner] += 1

print("Core features created")

Core features created


In [6]:
matches['win_rate_diff'] = matches['team1_win_rate'] - matches['team2_win_rate']
matches['form_diff'] = matches['team1_recent_form'] - matches['team2_recent_form']

print("Advanced features added")

Advanced features added


In [7]:
matches[['team1_win_rate','team2_win_rate',
         'team1_recent_form','team2_recent_form',
         'head_to_head',
         'win_rate_diff','form_diff']].tail(10)

,team1_win_rate,team2_win_rate,team1_recent_form,team2_recent_form,head_to_head,win_rate_diff,form_diff
881,0.642857,0.581395,0.4,0.4,0.0,0.061462,0.0
882,0.500000,0.554113,0.6,0.2,-13.0,-0.054113,0.4
883,0.494949,0.578704,0.6,0.4,-2.0,-0.083754,0.2
884,0.441441,0.550000,0.6,0.4,-2.0,-0.108559,0.2
885,0.492462,0.442396,0.4,0.4,5.0,0.050066,0.0
886,0.536585,0.551724,0.2,0.2,3.0,-0.015139,0.0
887,0.444954,0.466667,0.6,0.6,-8.0,-0.021713,0.0
888,0.469880,0.502242,0.6,0.8,-8.0,-0.032363,-0.2
889,0.467066,0.490000,0.6,0.2,1.0,-0.022934,0.4
890,0.470238,0.504464,0.6,1.0,-9.0,-0.034226,-0.4


In [8]:
matches['toss_winner_is_team1'] = (matches['toss_winner'] == matches['team1']).astype(int)

In [9]:
# 🔥 STRONG SIGNAL FEATURES

matches['win_rate_ratio'] = matches['team1_win_rate'] / (matches['team2_win_rate'] + 1e-5)
matches['form_ratio'] = matches['team1_recent_form'] / (matches['team2_recent_form'] + 1e-5)

# Clipped head-to-head (avoid extreme values)
matches['h2h_clipped'] = matches['head_to_head'].clip(-5, 5)

print("✅ Strong features added")

✅ Strong features added


In [15]:
features = matches[[
    'team1',
    'team2',
    
    'toss_winner_is_team1',
    
    'team1_win_rate',
    'team2_win_rate',
    'team1_recent_form',
    'team2_recent_form',
    
    'head_to_head',
    
    'win_rate_diff',
    'form_diff'
]]

target = matches['team1_win']

In [16]:
# ✅ ENCODE TEAM COLUMNS (IMPORTANT FIX)

features = pd.get_dummies(
    features,
    columns=['team1', 'team2'],
    drop_first=True
)

features = features.fillna(0)
features = features.astype(int)

print("Features shape:", features.shape)
features.head()

Features shape: (891, 26)


,toss_winner_is_team1,team1_win_rate,team2_win_rate,team1_recent_form,team2_recent_form,head_to_head,win_rate_diff,form_diff,team1_delhi capitals,team1_gujarat titans,...,team1_sunrisers hyderabad,team2_delhi capitals,team2_gujarat titans,team2_kolkata knight riders,team2_lucknow super giants,team2_mumbai indians,team2_punjab kings,team2_rajasthan royals,team2_royal challengers bangalore,team2_sunrisers hyderabad
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [12]:
print(features.dtypes)

toss_winner_is_team1                   int64
team1_win_rate                       float64
team2_win_rate                       float64
team1_recent_form                    float64
team2_recent_form                    float64
win_rate_diff                        float64
form_diff                            float64
win_rate_ratio                       float64
form_ratio                           float64
h2h_clipped                          float64
team1_delhi capitals                    bool
team1_gujarat titans                    bool
team1_kolkata knight riders             bool
team1_lucknow super giants              bool
team1_mumbai indians                    bool
team1_punjab kings                      bool
team1_rajasthan royals                  bool
team1_royal challengers bangalore       bool
team1_sunrisers hyderabad               bool
team2_delhi capitals                    bool
team2_gujarat titans                    bool
team2_kolkata knight riders             bool
team2_luck

In [13]:
# ✅ FINAL FIX — FORCE NUMERIC

features = features.fillna(0)

# Convert bool → int
features = features.astype(int)

print("Final dtypes:\n", features.dtypes)
print("Any object dtype:", (features.dtypes == 'object').any())

Final dtypes:
 toss_winner_is_team1                 int64
team1_win_rate                       int64
team2_win_rate                       int64
team1_recent_form                    int64
team2_recent_form                    int64
win_rate_diff                        int64
form_diff                            int64
win_rate_ratio                       int64
form_ratio                           int64
h2h_clipped                          int64
team1_delhi capitals                 int64
team1_gujarat titans                 int64
team1_kolkata knight riders          int64
team1_lucknow super giants           int64
team1_mumbai indians                 int64
team1_punjab kings                   int64
team1_rajasthan royals               int64
team1_royal challengers bangalore    int64
team1_sunrisers hyderabad            int64
team2_delhi capitals                 int64
team2_gujarat titans                 int64
team2_kolkata knight riders          int64
team2_lucknow super giants           in

In [14]:
features.to_csv("../data/feature_store/smart_features.csv", index=False)
target.to_csv("../data/feature_store/smart_target.csv", index=False)

print("FINAL CLEAN DATASET SAVED")

FINAL CLEAN DATASET SAVED
